# Mini-Project 5: Taxi Route Revenue, Pricing & Fleet Planner
**Scenario.** A taxi (matatu) association runs 14-seater vehicles on three routes out of Kampala. It wants to forecast demand, understand fares and decide how many vehicles to deploy. Ten days of passenger counts are given (the Ntinda figures come from the original question; the others are illustrative).

Logic lives in `src/taxi.py` (`Route`, `SupplyDemandModel`, three forecasters, `WalkForwardBacktester`, `FleetPlanner`). The forecasters **subclass the abstract `Forecaster` from Project 1** and reuse its `LinearTrendForecaster` unchanged ("days" play the role of "years"). Tests are in `tests/test_taxi.py`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.taxi import (
    ExponentialSmoothingForecaster, FleetPlanner, Forecaster, LinearTrendForecaster, MovingAverageForecaster,
    Route, SeasonalNaiveForecaster, SupplyDemandModel, WalkForwardBacktester, generate_weekly_demand,
    plot_revenue_forecasts, plot_seasonal_comparison,
)

SEED = 42
Path("figures").mkdir(exist_ok=True)
routes = [
    Route("Kampala-Ntinda",  [35, 40, 42, 50, 55, 60, 48, 52, 47, 45], fare=2000),
    Route("Kampala-Entebbe", [60, 58, 65, 70, 72, 80, 75, 68, 66, 64], fare=5000),
    Route("Kampala-Mukono",  [45, 47, 50, 49, 55, 62, 58, 53, 51, 50], fare=3000),
]
routes

## Task 1: `Route`: revenue and descriptive statistics

In [ ]:
print(f"{'route':<17}{'fare':>7}{'mean':>7}{'median':>8}{'variance':>10}{'st.dev':>8}{'min':>5}{'max':>5}{'total revenue (UGX)':>21}")
for r in routes:
    d = r.describe()
    print(f"{r.name:<17}{r.fare:7,.0f}{d['mean']:7.1f}{d['median']:8.1f}{d['variance']:10.2f}{d['stdev']:8.2f}{d['min']:5.0f}{d['max']:5.0f}{r.total_revenue():21,.0f}")
print("\nNtinda daily revenue (UGX):", routes[0].daily_revenue().astype(int).tolist())

Variance and standard deviation come from the `statistics` module (sample versions, divide by n − 1). Entebbe earns about 3.6× Ntinda's revenue with only 1.4× the passengers because its fare is 2.5× higher. Revenue is driven by fare × passengers, not passengers alone.

## Task 2: Supply–demand equilibrium for Ntinda
Demand Qd = 120 − 0.02P and supply Qs = 10 + 0.03P, with P the fare in UGX and Q passengers per trip-hour. Setting Qd = Qs = Q gives a 2×2 linear system in the unknowns (Q, P):

Q + 0.02·P = 120
Q − 0.03·P = 10

In [ ]:
model = SupplyDemandModel(demand_intercept=120, demand_slope=0.02, supply_intercept=10, supply_slope=0.03)
A, rhs = model.system()
print("A =\n", A, "\nrhs =", rhs)
price_eq, qty_eq = model.equilibrium()            # scipy.linalg.solve(A, rhs)
print(f"\nEquilibrium fare = UGX {price_eq:,.0f}, quantity = {qty_eq:.0f} passengers per trip-hour")
print("hand check: subtract the equations -> 0.05P = 110 -> P = 2200; Q = 120 - 0.02*2200 = 76")

fare = routes[0].fare
print(f"\nCurrent fare UGX {fare:,.0f} is {model.compare_fare(fare).upper()} equilibrium (by UGX {price_eq - fare:,.0f}, {(price_eq - fare)/price_eq:.1%})")
print(f"At UGX {fare:,}: demand = {model.demand(fare):.0f}, supply = {model.supply(fare):.0f}, excess demand = {model.excess_demand(fare):.0f} passengers per trip-hour")
rev_now, rev_eq = model.traded_quantity(fare) * fare, qty_eq * price_eq
print(f"Revenue per trip-hour: UGX {rev_now:,.0f} at the current fare (short side serves {model.traded_quantity(fare):.0f}) vs UGX {rev_eq:,.0f} at equilibrium ({rev_eq/rev_now - 1:+.1%})")

**Interpretation.** The current fare of UGX 2,000 is **below** the equilibrium fare of UGX 2,200 (about 9% lower). At that fare 80 passengers per trip-hour want a ride but vehicles supply seats for only 70: a **shortage of 10 passengers per trip-hour**, which shows up as queues, overcrowded vehicles and waiting. The market would clear at UGX 2,200 with 76 passengers served. Because the short side limits sales, the association serves 70 passengers at UGX 2,000 (UGX 140,000 per trip-hour) against 76 at UGX 2,200 (UGX 167,200), roughly 19% more revenue. **Caveats:** the curves are given, not estimated; Q is per trip-hour whereas the daily counts above are per day; fares on commuter routes are often regulated or constrained by competitors, so the association may not be free to move to the equilibrium fare.

## Task 3: `Forecaster` hierarchy
`MovingAverageForecaster` (window 3, the original method), `ExponentialSmoothingForecaster` (tunable α) and the reused `LinearTrendForecaster` all implement the abstract `fit()` / `predict(horizon)` interface from Project 1; `SeasonalNaiveForecaster` is added for the extension.

In [ ]:
for cls in (MovingAverageForecaster, ExponentialSmoothingForecaster, LinearTrendForecaster, SeasonalNaiveForecaster):
    print(f"{cls.__name__:<32} is a Forecaster: {issubclass(cls, Forecaster)}")
ntinda = routes[0]
for m in (MovingAverageForecaster(3), ExponentialSmoothingForecaster(0.5), LinearTrendForecaster()):
    print(f"{m.name:<24} day-11 forecast for Ntinda from all 10 days: {m.clone().fit(ntinda.days, ntinda.passengers).predict(1)[0]:.1f}")

## Task 4: Walk-forward (rolling-origin) backtest over days 4–10, with α tuned by grid search
For each target day *t* = 4…10 the model is fitted on days 1…*t*−1 only and forecasts day *t* (one step ahead), so no future data leaks into any forecast. MAE (passengers) is averaged over the 7 targets. α is tuned on a grid 0.05, 0.10, …, 1.00 by the same walk-forward MAE.

In [ ]:
bt = WalkForwardBacktester(first_target=4)            # targets: days 4..10
best_alpha, alpha_tables = {}, {}
for r in routes:
    best_alpha[r.name], alpha_tables[r.name] = bt.tune_alpha(r.days, r.passengers)

print("MAE (passengers) of SES at selected alphas:")
print(f"{'route':<17}" + "".join(f"{a:>7}" for a in (0.05, 0.25, 0.5, 0.75, 0.95, 1.0)) + f"{'best alpha':>12}")
for r in routes:
    t = alpha_tables[r.name]
    print(f"{r.name:<17}" + "".join(f"{t[a]:7.2f}" for a in (0.05, 0.25, 0.5, 0.75, 0.95, 1.0)) + f"{best_alpha[r.name]:12.2f}")

print("\nWalk-forward MAE per model per route (passengers; 7 one-step forecasts each)")
results, best_model = {}, {}
header = ["3-day moving average", "SES (tuned alpha)", "Linear trend"]
print(f"{'route':<17}" + "".join(f"{h:>22}" for h in header) + f"{'best':>24}")
for r in routes:
    models = [MovingAverageForecaster(3), ExponentialSmoothingForecaster(best_alpha[r.name]), LinearTrendForecaster()]
    res = [bt.evaluate_route(m, r) for m in models]
    results[r.name] = res
    best = min(zip(models, res), key=lambda mr: mr[1].mae)
    best_model[r.name] = best
    print(f"{r.name:<17}" + "".join(f"{x.mae:22.2f}" for x in res) + f"{best[1].model:>24}")

**Reading the results.**
- **SES beats the moving average and the linear trend on every route** (MAE 3.7–5.9 versus 5.3–7.5 and 6.4–7.9 passengers).
- **But the grid search chose α = 1.0 on all three routes, the edge of the grid.** With α = 1 the "smoothed" level is just yesterday's value, i.e. the naive forecast "tomorrow = today". So on these short, noisy series exponential *smoothing* adds nothing: the lowest error comes from not smoothing at all.
- **The linear trend does worst** because it extrapolates the rise on days 1–6 and misses the fall after the day-6 peak.
- **Caution: tuning α on the same days used for scoring flatters SES.** A cleaner check follows.

In [ ]:
print("Honest check: tune alpha on targets 4-7 only, then score every model on the unseen targets 8-10.")
print(f"{'route':<17}{'alpha (4-7)':>12}{'SES':>8}{'3-day MA':>10}{'Linear':>8}")
tune, score = WalkForwardBacktester(4, 7), WalkForwardBacktester(8, 10)
for r in routes:
    a, _ = tune.tune_alpha(r.days, r.passengers)
    ses = score.evaluate_route(ExponentialSmoothingForecaster(a), r).mae
    ma = score.evaluate_route(MovingAverageForecaster(3), r).mae
    lin = score.evaluate_route(LinearTrendForecaster(), r).mae
    print(f"{r.name:<17}{a:12.2f}{ses:8.2f}{ma:10.2f}{lin:8.2f}")

The ranking (SES < moving average < linear trend) survives on days 8–10, so the conclusion is not just an artefact of tuning and scoring on the same data. With only three held-out days, though, it is weak evidence.

## Task 5: Day-11 revenue forecast with the best model

In [ ]:
day11_pax, day11_rev = {}, {}
print(f"{'route':<17}{'best model':<18}{'day-11 passengers':>18}{'fare':>8}{'forecast revenue (UGX)':>24}{'avg daily revenue':>19}")
for r in routes:
    model = best_model[r.name][0]
    day11_pax[r.name] = float(model.clone().fit(r.days, r.passengers).predict(1)[0])
    day11_rev[r.name] = day11_pax[r.name] * r.fare
    print(f"{r.name:<17}{model.name:<18}{day11_pax[r.name]:18.1f}{r.fare:8,.0f}{day11_rev[r.name]:24,.0f}{r.daily_revenue().mean():19,.0f}")

backtests = {r.name: best_model[r.name][1] for r in routes}
fig = plot_revenue_forecasts(routes, backtests, day11_pax)
fig.savefig("figures/project5_revenue_forecast.png", dpi=150)
plt.show()

Because the winning model has α = 1, the day-11 forecast simply equals day 10's passenger count (45, 64 and 50). The chart shows why that is a reasonable but modest forecast: the red one-step forecasts always lag the actual series by a day and miss turning points, and the star is one more of those lagging points.

## Task 6: Fleet planning for day 11
Each vehicle makes 8 one-way trips a day carrying 14 passengers, so its capacity is 8 × 14 = **112 passengers a day**. Vehicles needed = ⌈forecast × 1.15 ÷ 112⌉ (15% buffer), rounded **up** because rounding down would leave passengers without a seat, with a minimum of one vehicle for a route that is operating.

In [ ]:
planner = FleetPlanner(trips_per_vehicle=8, seats=14, buffer=0.15)
print(f"capacity per vehicle = {planner.capacity_per_vehicle} passengers/day\n")
print(f"{'route':<17}{'forecast pax':>13}{'with 15% buffer':>16}{'vehicles (exact)':>17}{'vehicles':>9}{'utilisation':>12}{'max observed':>13}")
for r in routes:
    f = day11_pax[r.name]
    n = planner.vehicles_needed(f)
    print(f"{r.name:<17}{f:13.1f}{f*1.15:16.1f}{f*1.15/112:17.2f}{n:9d}{planner.utilisation(f, n):12.0%}{r.passengers.max():13.0f}")
print(f"\nA second vehicle is needed only above {112/1.15:.0f} forecast passengers a day (highest ever observed: {max(r.passengers.max() for r in routes):.0f}).")
print("Friday-style stress test (+30% demand) on Entebbe:", planner.vehicles_needed(day11_pax['Kampala-Entebbe'] * 1.3), "vehicle(s)")

**Result: one vehicle per route**, at 40–57% utilisation. A single 14-seater can carry 112 passengers a day, more than any route has ever drawn (60, 80 and 62 at peak), so a 15% buffer does not tip any route into a second vehicle, even with a 30% demand surge. **Caveat:** daily totals hide *when* people travel. If demand is concentrated in a morning and evening peak, one vehicle may not be enough at those hours even though it is half empty over the day; this needs hourly counts to resolve.

## Extension: 60 simulated days with a weekly pattern: seasonal naive vs moving average
Seeded simulation (day 1 = Monday): base level 50 passengers, multipliers Mon–Wed 1.0, Thu 1.05, **Fri 1.30**, Sat 1.10, **Sun 0.70**, plus Gaussian noise (sd 3). The seasonal-naive forecaster predicts "the same weekday last week". For a fair comparison, SES's α is tuned on targets 8–28 only, and **all models are scored on the unseen days 29–60**.

In [ ]:
y = generate_weekly_demand(n_days=60, base=50, seed=SEED, noise_sd=3.0)
days = np.arange(1, 61)
names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
print("mean passengers by weekday:", {n: round(float(y[i::7].mean()), 1) for i, n in enumerate(names)})

alpha_sim, _ = WalkForwardBacktester(8, 28).tune_alpha(days, y)
sim_models = [MovingAverageForecaster(3), ExponentialSmoothingForecaster(alpha_sim), LinearTrendForecaster(), SeasonalNaiveForecaster(7)]
score = WalkForwardBacktester(29, 60)
sim_results = {m.name: score.evaluate(m, days, y) for m in sim_models}

print(f"\n(alpha for SES tuned on days 8-28: {alpha_sim})")
print(f"{'model':<28}{'MAE':>7}{'RMSE':>7}")
for name, res in sorted(sim_results.items(), key=lambda kv: kv[1].mae):
    print(f"{name:<28}{res.mae:7.2f}{res.rmse:7.2f}")
ma_mae, sn_mae = sim_results["3-day moving average"].mae, sim_results["Seasonal naive (period 7)"].mae
print(f"\nSeasonal naive cuts the moving average's error by {1 - sn_mae/ma_mae:.0%} (MAE {sn_mae:.2f} vs {ma_mae:.2f}).")

fig = plot_seasonal_comparison(y, {k: sim_results[k] for k in ("3-day moving average", "Seasonal naive (period 7)")}, start_day=29)
fig.savefig("figures/project5_seasonal.png", dpi=150)
plt.show()

**Interpretation.** The seasonal-naive forecaster has the lowest error, roughly 60% below the 3-day moving average. The moving average averages across the weekly cycle, so it forecasts a middling level, too low on Fridays and too high on Sundays; the seasonal-naive forecast has learned the weekday shape. Its remaining error is mostly noise (it copies last week's value including last week's noise), so a seasonal average over several weeks would likely do slightly better. SES and the linear trend also fail for the same reason as the moving average: they have no notion of a weekly cycle. **Caveat:** the data are simulated with exactly the pattern the seasonal model assumes, so this demonstrates *when* seasonal methods win, not that real matatu demand has a 7-day pattern; real daily data would be needed to confirm it.

## Findings & Limitations
**Findings.** Over ten days the routes earned UGX 0.95 m (Ntinda), 3.39 m (Entebbe) and 1.56 m (Mukono): Entebbe leads mainly because of its higher fare. Solving the supply–demand system gives an equilibrium Ntinda fare of UGX 2,200 at 76 passengers per trip-hour, so the current UGX 2,000 fare is about 9% below it, leaving a shortage of 10 passengers per trip-hour. In walk-forward backtesting over days 4–10, exponential smoothing beat the 3-day moving average and the linear trend on every route (MAE 3.7–5.9 versus 5.3–7.5 and 6.4–7.9). However, the grid search chose α = 1.0 everywhere, which is just "tomorrow equals today", so smoothing adds nothing on these short, noisy series; an honest holdout (tune on days 4–7, score on 8–10) preserves the ranking. Day-11 forecasts are 45, 64 and 50 passengers, and one vehicle per route suffices (40–57% utilisation; a second is needed only above about 97 passengers a day). On simulated weekly data the seasonal-naive method cut error by about 60% relative to the moving average.

**Limitations.** Ten days and seven test targets make every ranking fragile. Daily totals hide peak-hour crowding, so one vehicle per route may be too few at peak times. The demand and supply curves are given, not estimated, and use per-trip-hour units unlike the daily counts. Only the Ntinda data are from the original question, and the weekly-pattern data are synthetic.